In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("results/dcama.csv")

In [3]:
def filter_dataframe(df, filter_dict):
    # Start with all True mask
    mask = pd.Series(True, index=df.index)
    
    # Add each filter condition
    for column, value in filter_dict.items():
        mask = mask & (df[column] == value)
    
    # Apply the mask to get filtered data
    return df[mask]

In [4]:
hyperparameters_cols = ["lora_r", "lr"]
hyperparameters = df[hyperparameters_cols].drop_duplicates()
hyperparameters

,lora_r,lr
0,64,0.0010
1,32,0.0010
2,16,0.0010
9,64,0.0001
13,32,0.0001
15,16,0.0001


In [5]:
def get_results(df, hyperparameter_dict):
    selected_cols = ["miou_orig"]
    filtered_df = filter_dataframe(df, hyperparameter_dict)
    num_iterations = filtered_df["num_iterations"].max()
    gain_columns = [f"gain_it_{i}" for i in range(1, num_iterations)]
    assert len(filtered_df) == 4, f"Expected 4 results, got {len(filtered_df)}"
    selected_df = filtered_df[selected_cols + gain_columns]
    mean_df = selected_df.mean()
    # Get the column where the mean is the highest
    best_column = mean_df[gain_columns].idxmax()
    best_gain = mean_df[best_column]
    
    final_df = filtered_df[[best_column, "val_fold_idx"]].set_index("val_fold_idx")[[best_column]].T
    final_df["mean"] = best_gain
    final_df["best_iteration"] = int(best_column.split("_")[-1])
    for key, value in hyperparameter_dict.items():
        final_df[key] = value
    final_df = final_df.reset_index(drop=True)
    # Remove val_fold_idx column
    return pd.DataFrame(final_df.to_dict())

dfs = []
for hyperparameter_dict in hyperparameters.to_dict(orient="records"):
    dfs.append(get_results(df, hyperparameter_dict))
results = pd.concat(dfs, ignore_index=True)
results

,2,3,0,1,mean,best_iteration,lora_r,lr
0,0.089515,0.055824,0.027766,0.022268,0.048843,4,64,0.0010
1,0.082705,0.051901,0.025588,0.018378,0.044643,4,32,0.0010
2,0.071264,0.052055,0.026545,0.018150,0.042003,4,16,0.0010
3,0.023902,0.013940,-0.006443,-0.003249,0.007037,4,64,0.0001
4,0.014711,0.006877,-0.010657,-0.005353,0.001394,4,32,0.0001
5,0.009952,0.004076,-0.012637,-0.010971,-0.002395,4,16,0.0001


# Pascal N1K5

In [117]:
import pandas as pd 
import wandb
api = wandb.Api()

filters = {
  "config.dataset": "pascal",
  "config.n_ways": 1,
  "config.val_num_samples": 5000
}
runs = api.runs("pasqualedem/lorafss", filters=filters)

runs_list = []
for run in runs: 
    runs_list.append({
        **run.summary._json_dict,
        **{k: v for k,v in run.config.items()
          if not k.startswith('_')},
        **{"name": run.name}
        })

runs_df = pd.DataFrame(runs_list)
runs_df


,_runtime,_step,_timestamp,_wandb.runtime,best_gain,best_miou,gain,gain_it_1,gain_it_2,gain_it_3,...,experiment,n_ways,device,target_modules,model,val_fold_idx,num_iterations,dataset,k_shots,name
0,57790.622019,31,1.737095e+09,57790,0.026196,0.705797,0.026196,0.008502,0.012662,0.015438,...,{'group': 'LoRa'},1,cuda,"[conv1, conv3]",bam,0,8,pascal,5,silver-meadow-215
1,61090.547890,31,1.737020e+09,61090,0.008480,0.714687,-0.002105,0.006811,0.007661,0.006492,...,{'group': 'LoRa'},1,cuda,[qkv],dcama,0,8,pascal,5,fancy-rain-214
2,60774.897256,31,1.737020e+09,60774,0.152522,0.680791,0.152522,0.053867,0.080869,0.105722,...,{'group': 'LoRa'},1,cuda,[qkv],dcama,2,8,pascal,5,kind-sponge-213
3,60981.457984,31,1.737020e+09,60981,0.045463,0.667366,0.045463,0.022482,0.029463,0.039560,...,{'group': 'LoRa'},1,cuda,[qkv],dcama,3,8,pascal,5,proud-puddle-212
4,61010.818042,31,1.737020e+09,61010,0.007750,0.718181,0.007546,0.004023,0.006669,0.006215,...,{'group': 'LoRa'},1,cuda,[qkv],dcama,1,8,pascal,5,valiant-fire-211
5,57795.871520,31,1.737017e+09,57795,0.036680,0.695652,0.036680,0.017686,0.028450,0.033169,...,{'group': 'LoRa'},1,cuda,"[conv1, conv3]",bam,2,8,pascal,5,divine-bird-210
6,57975.966248,31,1.737017e+09,57975,0.041208,0.659962,0.040176,0.029904,0.039495,0.041208,...,{'group': 'LoRa'},1,cuda,"[conv1, conv3]",bam,3,8,pascal,5,desert-elevator-208
7,57878.559391,31,1.737017e+09,57878,0.001432,0.715625,-0.008629,0.001432,-0.001830,-0.005267,...,{'group': 'LoRa'},1,cuda,"[conv1, conv3]",bam,1,8,pascal,5,lucky-star-207


In [118]:
cols = ["model", "miou_orig", "miou_it_7", "gain_it_4", "best_gain"]

runs_df[cols].groupby("model").mean()

,miou_orig,miou_it_7,gain_it_4,best_gain
model,,,,
bam,0.667880,0.691486,0.021700,0.026379
dcama,0.641702,0.692559,0.044135,0.053554


In [119]:
orig = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_orig")
tap  = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_it_7").rename(index=lambda x: f"TaP ({x})")
result = pd.concat([orig, tap]).reset_index()
result["mean"] = result[[0, 1, 2, 3]].mean(axis=1)
result = result.round(3)
result.columns.name = "index"
result

index,model,0,1,2,3,mean
0,bam,0.680,0.714,0.659,0.619,0.668
1,dcama,0.706,0.710,0.528,0.622,0.642
2,TaP (bam),0.706,0.706,0.696,0.659,0.691
3,TaP (dcama),0.704,0.718,0.681,0.667,0.693


In [120]:
result.to_csv("results/Pascal_N1K5.csv", index=False)

# Pascal N2K5

In [121]:
import pandas as pd 
import wandb
api = wandb.Api()

filters = {
  "config.dataset": "pascal",
  "config.n_ways": 2,
}
runs = api.runs("pasqualedem/lorafss", filters=filters)

runs_list = []
for run in runs: 
    runs_list.append({
        **run.summary._json_dict,
        **{k: v for k,v in run.config.items()
          if not k.startswith('_')},
        **{"name": run.name}
        })

runs_df = pd.DataFrame(runs_list)
runs_df


,_runtime,_step,_timestamp,_wandb.runtime,best_gain,best_miou,gain,gain_it_1,gain_it_2,gain_it_3,...,model,lr,dataset,lora_dropout,lora_r,k_shots,num_iterations,val_num_samples,n_ways,name
0,43087.606206,31,1.737081e+09,43087,0.075502,0.562095,0.075502,0.033003,0.034542,0.040871,...,bam,0.0001,pascal,0.1,64,5,8,1000,2,dulcet-star-219
1,43142.839254,31,1.737081e+09,43142,0.068192,0.625605,0.068192,0.030990,0.041121,0.050720,...,bam,0.0001,pascal,0.1,64,5,8,1000,2,bright-breeze-218
2,43188.283762,31,1.737081e+09,43188,0.052871,0.576545,0.052871,0.025731,0.025996,0.031364,...,bam,0.0001,pascal,0.1,64,5,8,1000,2,fine-water-217
3,43096.431167,31,1.737081e+09,43096,0.143347,0.549734,0.143347,0.091061,0.105579,0.111546,...,bam,0.0001,pascal,0.1,64,5,8,1000,2,light-oath-216
4,55753.917659,31,1.737017e+09,55753,0.087736,0.529018,0.087736,0.038246,0.049516,0.061981,...,dcama,0.0010,pascal,0.1,64,5,8,1000,2,vital-sunset-206
5,52176.604356,31,1.737013e+09,52176,0.121879,0.426497,0.121879,0.036450,0.051080,0.071890,...,dcama,0.0010,pascal,0.1,64,5,8,1000,2,expert-breeze-205
6,52836.558201,31,1.737014e+09,52836,0.054702,0.579321,0.027823,0.037868,0.027497,0.030906,...,dcama,0.0010,pascal,0.1,64,5,8,1000,2,confused-puddle-204
7,51144.651483,31,1.737012e+09,51144,0.044386,0.511105,0.006460,0.023636,0.026763,0.031733,...,dcama,0.0010,pascal,0.1,64,5,8,1000,2,wandering-fire-202


In [122]:
cols = ["model", "miou_orig", "val_fold_idx", "miou_it_7"]
runs_df[cols]

,model,miou_orig,val_fold_idx,miou_it_7
0,bam,0.486593,1,0.562095
1,bam,0.557413,2,0.625605
2,bam,0.523674,3,0.576545
3,bam,0.406386,0,0.549734
4,dcama,0.441282,2,0.529018
5,dcama,0.304618,3,0.426497
6,dcama,0.524619,1,0.552442
7,dcama,0.466719,0,0.473179


In [123]:
runs_df[["model", "val_fold_idx", "miou_orig"]]

,model,val_fold_idx,miou_orig
0,bam,1,0.486593
1,bam,2,0.557413
2,bam,3,0.523674
3,bam,0,0.406386
4,dcama,2,0.441282
5,dcama,3,0.304618
6,dcama,1,0.524619
7,dcama,0,0.466719


In [124]:
orig = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_orig")
tap  = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_it_7").rename(index=lambda x: f"TaP ({x})")
result = pd.concat([orig, tap]).reset_index()
result["mean"] = result[[0, 1, 2, 3]].mean(axis=1)
result = result.round(3)
result.columns.name = "index"
result

index,model,0,1,2,3,mean
0,bam,0.406,0.487,0.557,0.524,0.494
1,dcama,0.467,0.525,0.441,0.305,0.434
2,TaP (bam),0.550,0.562,0.626,0.577,0.578
3,TaP (dcama),0.473,0.552,0.529,0.426,0.495


In [125]:
result.to_csv("results/Pascal_N2K5.csv", index=False)

# COCO N1K5

In [19]:
import pandas as pd 
import wandb
api = wandb.Api()

filters = {
  "config.dataset": "coco",
  "config.n_ways": 1,
}
runs = api.runs("pasqualedem/lorafss", filters=filters)

runs_list = []
for run in runs: 
    runs_list.append({
        **run.summary._json_dict,
        **{k: v for k,v in run.config.items()
          if not k.startswith('_')},
        **{"name": run.name}
        })

runs_df = pd.DataFrame(runs_list)
runs_df


,_runtime,_step,_timestamp,_wandb.runtime,best_gain,best_miou,gain,gain_it_1,gain_it_2,gain_it_3,...,model,lr,k_shots,device,experiment,dataset,num_iterations,n_ways,name,k_shot
0,39698.515302,31,1.738122e+09,39698,0.000000,0.622380,-0.008340,-0.005458,-0.006295,-0.003785,...,hdmnet,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,elated-butterfly-259,NaN
1,39703.173335,31,1.738122e+09,39703,0.009170,0.559849,0.008472,0.009170,0.009119,0.008263,...,hdmnet,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,swift-brook-258,NaN
2,39726.449160,31,1.738122e+09,39726,0.030612,0.548673,0.029315,0.010105,0.021754,0.028110,...,hdmnet,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,desert-frog-257,NaN
3,39678.804412,31,1.738122e+09,39678,0.037096,0.567172,0.037096,0.020496,0.030268,0.035827,...,hdmnet,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,northern-plasma-256,NaN
4,57955.585230,31,1.733291e+09,57955,0.079929,0.567667,0.079929,0.043999,0.065549,0.068814,...,bam,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,royal-planet-194,5.0
5,58066.997370,31,1.733292e+09,58067,0.067779,0.508862,0.067779,0.037484,0.043651,0.050009,...,bam,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,youthful-dream-193,5.0
6,58093.163524,31,1.733292e+09,58093,0.066204,0.588995,0.066204,0.028287,0.041955,0.044444,...,bam,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,comic-disco-192,5.0
7,57956.152316,31,1.733291e+09,57956,0.071753,0.536488,0.071753,0.050120,0.064235,0.063799,...,bam,0.0001,5,cuda,{'group': 'LoRa'},coco,8,1,giddy-wave-191,5.0
8,61138.160641,31,1.732704e+09,61138,0.011021,0.609441,0.011021,0.007817,0.008262,0.009067,...,dcama,0.0010,5,cuda,{'group': 'LoRa'},coco,8,1,proud-gorge-190,5.0
9,60975.203757,31,1.732704e+09,60975,0.016215,0.592479,0.013047,0.016215,0.015976,0.014736,...,dcama,0.0010,5,cuda,{'group': 'LoRa'},coco,8,1,morning-valley-189,5.0


In [20]:
run

<Run pasqualedem/lorafss/xnjaplm4 (finished)>

In [21]:
cols = ["model", "miou_orig", "val_fold_idx", "miou_it_7"]
runs_df[cols]

,model,miou_orig,val_fold_idx,miou_it_7
0,hdmnet,0.622380,1,0.614040
1,hdmnet,0.550679,3,0.559151
2,hdmnet,0.518061,0,0.547376
3,hdmnet,0.530077,2,0.567172
4,bam,0.487738,2,0.567667
5,bam,0.441083,0,0.508862
6,bam,0.522792,1,0.588995
7,bam,0.464735,3,0.536488
8,dcama,0.598420,2,0.609441
9,dcama,0.576264,1,0.589310


In [22]:
runs_df[["model", "val_fold_idx", "miou_orig"]]

,model,val_fold_idx,miou_orig
0,hdmnet,1,0.622380
1,hdmnet,3,0.550679
2,hdmnet,0,0.518061
3,hdmnet,2,0.530077
4,bam,2,0.487738
5,bam,0,0.441083
6,bam,1,0.522792
7,bam,3,0.464735
8,dcama,2,0.598420
9,dcama,1,0.576264


In [23]:
orig = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_orig")
tap  = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_it_7").rename(index=lambda x: f"TaP ({x})")
result = pd.concat([orig, tap]).reset_index()
result["mean"] = result[[0, 1, 2, 3]].mean(axis=1)
result = result.round(3)
result.columns.name = "index"
result

index,model,0,1,2,3,mean
0,bam,0.441,0.523,0.488,0.465,0.479
1,dcama,0.564,0.576,0.598,0.584,0.581
2,hdmnet,0.518,0.622,0.530,0.551,0.555
3,TaP (bam),0.509,0.589,0.568,0.536,0.551
4,TaP (dcama),0.598,0.589,0.609,0.595,0.598
5,TaP (hdmnet),0.547,0.614,0.567,0.559,0.572


In [24]:
result.to_csv("results/COCO_N1K5.csv", index=False)

# COCO N2K5

In [13]:
import pandas as pd 
import wandb
api = wandb.Api()

filters = {
  "config.dataset": "coco",
  "config.n_ways": 2,
  "config.val_num_samples": 1000,
  "summary_metrics.miou_it_7": {"$ne": None}  # Filter for non-null values
}
runs = api.runs("pasqualedem/lorafss", filters=filters)

runs_list = []
for run in runs: 
    runs_list.append({
        **run.summary._json_dict,
        **{k: v for k,v in run.config.items()
          if not k.startswith('_')},
        **{"name": run.name}
        })

runs_df = pd.DataFrame(runs_list)
runs_df


,_runtime,_step,_timestamp,_wandb.runtime,best_gain,best_miou,gain,gain_it_1,gain_it_2,gain_it_3,...,k_shots,dataset,lora_dropout,model,experiment,val_fold_idx,lora_r,device,lr,name
0,30507.061487,31,1.738113e+09,30507,0.082323,0.574612,0.082323,0.023162,0.074608,0.074440,...,5,coco,0.1,hdmnet,{'group': 'LoRa'},2,64,cuda,0.0001,breezy-water-255
1,30547.031542,31,1.738113e+09,30547,0.024685,0.539542,0.017012,0.024685,0.005270,0.012827,...,5,coco,0.1,hdmnet,{'group': 'LoRa'},3,64,cuda,0.0001,royal-leaf-254
2,30501.527660,31,1.738113e+09,30501,0.027461,0.602079,0.027461,-0.006689,0.016246,0.015310,...,5,coco,0.1,hdmnet,{'group': 'LoRa'},1,64,cuda,0.0001,happy-night-253
3,30534.786390,31,1.738113e+09,30534,0.032127,0.512031,0.032127,-0.023805,-0.006789,0.011611,...,5,coco,0.1,hdmnet,{'group': 'LoRa'},0,64,cuda,0.0001,lively-valley-252
4,51729.824789,31,1.738139e+09,51729,0.050638,0.523008,0.040134,0.036116,0.048032,0.050638,...,5,coco,0.1,dcama,{'group': 'LoRa'},3,64,cuda,0.0010,atomic-cloud-251
5,51442.762697,31,1.738139e+09,51442,0.079506,0.568182,0.079506,0.048164,0.049803,0.062485,...,5,coco,0.1,dcama,{'group': 'LoRa'},2,64,cuda,0.0010,brisk-firebrand-250
6,53386.592739,31,1.738141e+09,53386,0.068150,0.512083,0.068150,0.043744,0.049411,0.054695,...,5,coco,0.1,dcama,{'group': 'LoRa'},0,64,cuda,0.0010,genial-river-249
7,51554.881371,31,1.738139e+09,51554,0.041500,0.495375,0.029736,0.041500,0.025283,0.029188,...,5,coco,0.1,dcama,{'group': 'LoRa'},1,64,cuda,0.0010,silver-dragon-248


In [14]:
runs_df[cols]

,model,miou_orig,val_fold_idx,miou_it_7
0,hdmnet,0.492288,2,0.574612
1,hdmnet,0.514856,3,0.531868
2,hdmnet,0.574619,1,0.602079
3,hdmnet,0.479904,0,0.512031
4,dcama,0.472370,3,0.512504
5,dcama,0.488677,2,0.568182
6,dcama,0.443933,0,0.512083
7,dcama,0.453875,1,0.483611


In [15]:
cols = ["model", "miou_orig", "val_fold_idx", "miou_it_7"]
runs_df[cols]

,model,miou_orig,val_fold_idx,miou_it_7
0,hdmnet,0.492288,2,0.574612
1,hdmnet,0.514856,3,0.531868
2,hdmnet,0.574619,1,0.602079
3,hdmnet,0.479904,0,0.512031
4,dcama,0.472370,3,0.512504
5,dcama,0.488677,2,0.568182
6,dcama,0.443933,0,0.512083
7,dcama,0.453875,1,0.483611


In [16]:
runs_df[["model", "val_fold_idx", "miou_orig"]]

,model,val_fold_idx,miou_orig
0,hdmnet,2,0.492288
1,hdmnet,3,0.514856
2,hdmnet,1,0.574619
3,hdmnet,0,0.479904
4,dcama,3,0.472370
5,dcama,2,0.488677
6,dcama,0,0.443933
7,dcama,1,0.453875


In [17]:
orig = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_orig")
tap  = runs_df.pivot(index="model", columns="val_fold_idx", values="miou_it_7").rename(index=lambda x: f"TaP ({x})")
result = pd.concat([orig, tap]).reset_index()
result["mean"] = result[[0, 1, 2, 3]].mean(axis=1)
result = result.round(3)
result.columns.name = "index"
result

index,model,0,1,2,3,mean
0,dcama,0.444,0.454,0.489,0.472,0.465
1,hdmnet,0.480,0.575,0.492,0.515,0.515
2,TaP (dcama),0.512,0.484,0.568,0.513,0.519
3,TaP (hdmnet),0.512,0.602,0.575,0.532,0.555


In [18]:
result.to_csv("results/COCO_N2K5.csv", index=False)